# LIMPIEZA DATOS IPC ESPAÑA INE

In [1]:
import pandas as pd
import os

In [3]:
%pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
ruta_ipc = "datos/IPC_INE_ORIGINAL.xlsx"

IPC_ORIGINAL = pd.read_excel(ruta_ipc)

In [5]:
print("Dimensiones:", IPC_ORIGINAL.shape)

print("\nColumnas:")
print(IPC_ORIGINAL.columns.tolist())

print("\nPrimeras filas:")
display(IPC_ORIGINAL.head())

print("\nTipos de datos:")
print(IPC_ORIGINAL.dtypes)

Dimensiones: (202, 4)

Columnas:
['PERIODO', 'VALOR', 'TIPO DATO', 'NOTA']

Primeras filas:


,PERIODO,VALOR,TIPO DATO,NOTA
0,2026M10,0.6,Avance,NaN
1,2026M09,0.3,Definitivo,NaN
2,2026M08,0.7,Definitivo,NaN
3,2026M07,0.3,Definitivo,NaN
4,2026M06,0.6,Definitivo,NaN



Tipos de datos:
PERIODO          str
VALOR        float64
TIPO DATO        str
NOTA         float64
dtype: object


In [6]:
# Creamos una copia para trabajar sin modificar el Excel original
ipc = IPC_ORIGINAL.copy()

# Comprobamos que la copia se ha creado correctamente
display(ipc.head())

,PERIODO,VALOR,TIPO DATO,NOTA
0,2026M10,0.6,Avance,NaN
1,2026M09,0.3,Definitivo,NaN
2,2026M08,0.7,Definitivo,NaN
3,2026M07,0.3,Definitivo,NaN
4,2026M06,0.6,Definitivo,NaN


In [8]:
# Convertimos los periodos en fechas
ipc["fecha"] = pd.to_datetime(
    ipc["PERIODO"].str.replace("M", "-", regex=False) + "-01"
)

# Comprobamos el resultado
display(ipc[["PERIODO", "fecha"]].head())

,PERIODO,fecha
0,2026M10,2026-10-01
1,2026M09,2026-09-01
2,2026M08,2026-08-01
3,2026M07,2026-07-01
4,2026M06,2026-06-01


In [9]:
# Ordenamos los datos desde el mes más antiguo hasta el más reciente
ipc = ipc.sort_values("fecha").reset_index(drop=True)

# Comprobamos las primeras y últimas filas
print("Primeros registros:")
display(ipc.head())

print("Últimos registros:")
display(ipc.tail())

Primeros registros:


,PERIODO,VALOR,TIPO DATO,NOTA,fecha
0,2010M01,-1.0,Definitivo,NaN,2010-01-01
1,2010M02,-0.2,Definitivo,NaN,2010-02-01
2,2010M03,0.7,Definitivo,NaN,2010-03-01
3,2010M04,1.1,Definitivo,NaN,2010-04-01
4,2010M05,0.2,Definitivo,NaN,2010-05-01


Últimos registros:


,PERIODO,VALOR,TIPO DATO,NOTA,fecha
197,2026M06,0.6,Definitivo,NaN,2026-06-01
198,2026M07,0.3,Definitivo,NaN,2026-07-01
199,2026M08,0.7,Definitivo,NaN,2026-08-01
200,2026M09,0.3,Definitivo,NaN,2026-09-01
201,2026M10,0.6,Avance,NaN,2026-10-01


In [10]:
# Nos quedamos con los datos hasta septiembre de 2026
ipc = ipc[ipc["fecha"] <= "2026-09-01"].copy()

# Comprobamos el periodo final
print("Primer mes:", ipc["fecha"].min())
print("Último mes:", ipc["fecha"].max())
print("Número de registros:", len(ipc))

Primer mes: 2010-01-01 00:00:00
Último mes: 2026-09-01 00:00:00
Número de registros: 201


In [11]:
# Creamos la lista de todos los meses que deberían existir
meses_esperados = pd.date_range(
    start="2010-01-01",
    end="2026-09-01",
    freq="MS"
)

# Comprobamos si nuestros datos contienen todos esos meses
meses_faltantes = meses_esperados.difference(ipc["fecha"])

print("Meses esperados:", len(meses_esperados))
print("Meses disponibles:", len(ipc))
print("Meses que faltan:", len(meses_faltantes))
print(meses_faltantes)

Meses esperados: 201
Meses disponibles: 201
Meses que faltan: 0
DatetimeIndex([], dtype='datetime64[us]', freq='MS')


In [12]:
# Calculamos el factor acumulado de inflación
ipc["factor_ipc"] = (1 + ipc["VALOR"] / 100).cumprod()

# Establecemos enero de 2010 como base 100
ipc["indice_ipc"] = (
    ipc["factor_ipc"] / ipc["factor_ipc"].iloc[0]
) * 100

# Mostramos los resultados
display(ipc[["fecha", "VALOR", "indice_ipc"]].head(10))

,fecha,VALOR,indice_ipc
0,2010-01-01,-1.0,100.000000
1,2010-02-01,-0.2,99.800000
2,2010-03-01,0.7,100.498600
3,2010-04-01,1.1,101.604085
4,2010-05-01,0.2,101.807293
5,2010-06-01,0.2,102.010907
6,2010-07-01,-0.4,101.602864
7,2010-08-01,0.3,101.907672
8,2010-09-01,0.1,102.009580
9,2010-10-01,0.9,102.927666


In [13]:
print("Índice en enero de 2010:")
print(ipc["indice_ipc"].iloc[0])

print("\nÍndice en septiembre de 2026:")
print(ipc["indice_ipc"].iloc[-1])

print("\nEvolución acumulada del IPC desde enero de 2010:")
print(round(ipc["indice_ipc"].iloc[-1] - 100, 2), "%")

Índice en enero de 2010:
100.0

Índice en septiembre de 2026:
144.76607334704656

Evolución acumulada del IPC desde enero de 2010:
44.77 %


In [14]:
# Creamos la tabla definitiva del IPC
ipc_limpio = ipc[["fecha", "VALOR", "indice_ipc"]].copy()

# Renombramos las columnas
ipc_limpio = ipc_limpio.rename(columns={
    "VALOR": "ipc_mensual_pct"
})

# Comprobamos el resultado
display(ipc_limpio.head())
display(ipc_limpio.tail())

print("Dimensiones finales:", ipc_limpio.shape)

,fecha,ipc_mensual_pct,indice_ipc
0,2010-01-01,-1.0,100.000000
1,2010-02-01,-0.2,99.800000
2,2010-03-01,0.7,100.498600
3,2010-04-01,1.1,101.604085
4,2010-05-01,0.2,101.807293


,fecha,ipc_mensual_pct,indice_ipc
196,2026-05-01,0.1,142.048770
197,2026-06-01,0.6,142.901063
198,2026-07-01,0.3,143.329766
199,2026-08-01,0.7,144.333074
200,2026-09-01,0.3,144.766073


Dimensiones finales: (201, 3)


In [15]:
# Guardamos el IPC limpio en la carpeta datos
ipc_limpio.to_csv(
    "datos/ipc_espana_limpio.csv",
    index=False
)

print("Archivo IPC guardado correctamente.")

Archivo IPC guardado correctamente.
